In [ ]:
import sys
import re
import yaml
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pydeck as pdk

# 한글 폰트 및 마이너스 깨짐 방지
plt.rc('font', family='NanumGothic' if sys.platform == 'win32' else 'AppleGothic')
plt.rc('axes', unicode_minus=False)

# PROJECT_ROOT 자동 설정 (notebooks/ 기준 상위 폴더)
current_dir = Path.cwd()
PROJECT_ROOT = current_dir.parent if current_dir.name == 'notebooks' else current_dir

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# src 모듈 로드
from src.features import process_calendar_features, add_wind_features
from src.utils import calculate_metric, CAPACITY_KWH, TARGET_COLS

# 주요 디렉터리 경로 정의
INFO_DIR = PROJECT_ROOT / "information"
TRAIN_DIR = PROJECT_ROOT / "train"

print(f"📌 Project Root: {PROJECT_ROOT}")
print(f"📌 Info Directory: {INFO_DIR}")
print(f"📌 Train Directory: {TRAIN_DIR}")
print(f"✅ src.features 및 src.utils 모듈 로드 성공!")

In [ ]:
# 데이터 로드
train_labels = pd.read_csv(TRAIN_DIR / "train_labels.csv", encoding="utf-8-sig")
ldaps_train = pd.read_csv(TRAIN_DIR / "ldaps_train.csv", encoding="utf-8-sig")
gfs_train = pd.read_csv(TRAIN_DIR / "gfs_train.csv", encoding="utf-8-sig")

# 기초 결측치 및 데이터 형태 확인
print("=== 1. 데이터 Shape 및 결측치 현황 ===")
print(f"train_labels Shape : {train_labels.shape}")
print(f"ldaps_train  Shape : {ldaps_train.shape}")
print(f"gfs_train    Shape : {gfs_train.shape}\n")

# 데이터프레임 전체 결측치 총 개수 확인 (0이면 결측치 없음)
print("LDAPS 전체 결측치 총합:", ldaps_train.isnull().sum().sum())
print("GFS   전체 결측치 총합:", gfs_train.isnull().sum().sum())

# 결측치가 1개 이상 존재하는 컬럼만 필터링하여 출력
print("\n--- train_labels 결측 컬럼 ---")
print(train_labels.isnull().sum()[lambda x: x > 0])

print("\n--- ldaps_train 결측 컬럼 ---")
print(ldaps_train.isnull().sum()[lambda x: x > 0])

print("\n--- gfs_train 결측 컬럼 ---")
print(gfs_train.isnull().sum()[lambda x: x > 0])

In [ ]:
# 1. 달력 피처 생성
labels_df = train_labels.copy()
labels_df['forecast_kst_dtm'] = pd.to_datetime(labels_df['kst_dtm'])
cal_features = process_calendar_features(labels_df['forecast_kst_dtm'])
df_cal_analysis = pd.concat([labels_df, cal_features], axis=1)

# 2. 달력 피처와 타겟 간 피어슨 상관계수 분석
cal_cols = ['month', 'hour', 'dayofweek', 'is_weekend', 'hour_sin', 'hour_cos', 'month_sin', 'month_cos']
corr_cal = df_cal_analysis[cal_cols + TARGET_COLS].corr().loc[cal_cols, TARGET_COLS]

plt.figure(figsize=(8, 6))
sns.heatmap(corr_cal, annot=True, fmt=".4f", cmap="Blues")
plt.title("달력/시간 피처와 타겟 발전량 상관관계")
plt.show()

# 3. 시간별(Hour) 및 월별(Month) 평균 발전량 패턴 시각화
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# 시간별 발전량 평균
df_cal_analysis.groupby('hour')[TARGET_COLS].mean().plot(ax=axes[0], marker='o')
axes[0].set_title("시간대별(Hour) 평균 발전량")
axes[0].set_xlabel("Hour (0~23)")
axes[0].set_ylabel("평균 발전량 (kWh)")
axes[0].grid(True, alpha=0.3)

# 월별 발전량 평균
df_cal_analysis.groupby('month')[TARGET_COLS].mean().plot(ax=axes[1], marker='o')
axes[1].set_title("월별(Month) 평균 발전량")
axes[1].set_xlabel("Month (1~12)")
axes[1].set_ylabel("평균 발전량 (kWh)")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# information/info.xlsx 좌표(DMS -> Decimal) 및 KPX그룹 파싱
def parse_info_excel(filepath):
    info_raw = pd.read_excel(filepath, header=3).dropna(how='all', axis=1).dropna(how='all', axis=0)
    info_df = info_raw.copy()
    info_df['KPX그룹'] = info_df['KPX그룹'].ffill().astype(int)
    
    def dms_to_dd(dms_str):
        lat_m = re.search(r'(\d+)°(\d+)\'([\d.]+)"N', str(dms_str))
        lon_m = re.search(r'(\d+)°(\d+)\'([\d.]+)"E', str(dms_str))
        if not lat_m or not lon_m:
            return pd.Series([None, None])
        lat = float(lat_m.group(1)) + float(lat_m.group(2))/60 + float(lat_m.group(3))/3600
        lon = float(lon_m.group(1)) + float(lon_m.group(2))/60 + float(lon_m.group(3))/3600
        return pd.Series([lat, lon])

    info_df[['latitude', 'longitude']] = info_df['좌표(Google)'].apply(dms_to_dd)
    return info_df

info_df = parse_info_excel(INFO_DIR / "info.xlsx")

# 2. train 기상예보 및 Target Labels 로드
ldaps_train = pd.read_csv(TRAIN_DIR / "ldaps_train.csv", encoding='utf-8-sig')
gfs_train = pd.read_csv(TRAIN_DIR / "gfs_train.csv", encoding='utf-8-sig')
train_labels = pd.read_csv(TRAIN_DIR / "train_labels.csv", encoding='utf-8-sig')

# 3. main_train.py의 Grid Pivot + Mean 병합 로직
def process_weather_data(df, prefix):
    df = df.copy()
    df["forecast_kst_dtm"] = pd.to_datetime(df["forecast_kst_dtm"])
    drop_cols = {"data_available_kst_dtm", "latitude", "longitude"}
    value_cols = [c for c in df.columns if c not in {"forecast_kst_dtm", "grid_id", *drop_cols}]
    
    pivoted = df.pivot(index="forecast_kst_dtm", columns="grid_id", values=value_cols)
    pivoted.columns = [f"{prefix}_g{col[1]}_{col[0]}" for col in pivoted.columns]
    pivoted = pivoted.reset_index()
    
    agg_mean = df.groupby("forecast_kst_dtm")[value_cols].mean()
    agg_mean.columns = [f"{prefix}_mean_{c}" for c in agg_mean.columns]
    agg_mean = agg_mean.reset_index()
    
    return pivoted.merge(agg_mean, on="forecast_kst_dtm", how="inner")

train_weather = process_weather_data(ldaps_train, "ldaps").merge(
    process_weather_data(gfs_train, "gfs"), on="forecast_kst_dtm", how="inner"
)

# 4. 피처 생성 적용 (process_calendar_features & add_wind_features)
train_base = train_labels.rename(columns={"kst_dtm": "forecast_kst_dtm"})
train_base["forecast_kst_dtm"] = pd.to_datetime(train_base["forecast_kst_dtm"])
train_df = train_base.merge(train_weather, on="forecast_kst_dtm", how="left")

cal_feat = process_calendar_features(train_df["forecast_kst_dtm"])
X_train_raw = pd.concat([cal_feat, train_df.drop(columns=["forecast_kst_dtm", *TARGET_COLS])], axis=1)
X_train_feat = add_wind_features(X_train_raw)

# 전체 병합 데이터셋 구축
df_full = pd.concat([train_df[["forecast_kst_dtm", *TARGET_COLS]], X_train_feat], axis=1)

print(f"✅ 데이터 전처리 및 피처 생성 완료! Shape: {df_full.shape}")

✅ 데이터 전처리 및 피처 생성 완료! Shape: (26304, 1075)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, col in enumerate(TARGET_COLS):
    cap = CAPACITY_KWH[col]
    valid_data = df_full[col].dropna()
    
    sns.histplot(valid_data, bins=50, ax=axes[i], kde=True, color='navy')
    axes[i].axvline(cap * 0.1, color='crimson', linestyle='--', linewidth=2, label=f'FICR 10% 기준 ({cap*0.1:.0f} kWh)')
    
    zero_ratio = (valid_data == 0).mean() * 100
    ficr_eval_ratio = (valid_data >= cap * 0.1).mean() * 100
    
    axes[i].set_title(f"{col.upper()} 발전량 분포\n(0 kWh: {zero_ratio:.2f}% | FICR 평가대상: {ficr_eval_ratio:.2f}%)")
    axes[i].set_xlabel("Power Generation (kWh)")
    axes[i].legend()

plt.tight_layout()
plt.show()

# Group 3의 2022년 데이터 결측 구간 재확인
df_full['year'] = df_full['forecast_kst_dtm'].dt.year
print("=== 연도별 Target 유효 데이터 건수 ===")
print(df_full.groupby('year')[TARGET_COLS].count())

위 발전량 분포를 통해,
1. 0kwh 쏠림 현상 : 실제 발전량이 0인 비율이 전체의 11.4 ~ 16.2까지 차지
 - 일반적 회귀 모델은 0kwh근처에서 음수 예측값을 내놓을 수 있어, 추론 및 validation단계에서 하한선을 0으로 바운딩해주는 작업이 필수.
2. FiCR 평가 대상 구간(설비용량 10% 이상)
 - 실제 발전량이 설비용량의 10% 이상이 되는 구간은 전체의 약 53.7% ~ 60.7%
 - 즉, 바람이 아주 약하게 불어 이용률이 10% 미만인 나머지 40% 구간은 FICR 정산금 평가에서 아예 제외.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 11))

# 1. GFS 117m 풍속 vs Group 1 발전량 (파워 커브 S-Curve 형태 확인)
axes[0, 0].hexbin(
    df_full['gfs_mean_117m_ws'], 
    df_full['kpx_group_1'], 
    gridsize=50, 
    cmap='plasma', 
    mincnt=1
)
axes[0, 0].set_title("1) GFS 117m Wind Speed vs Group 1 Power Curve")
axes[0, 0].set_xlabel("117m Wind Speed (m/s)")
axes[0, 0].set_ylabel("Group 1 Power (kWh)")
axes[0, 0].grid(True, alpha=0.3)

# 2. 선형 풍속(v) vs 3제곱 풍속(v^3) 타겟 상관계수 비교
v_cols = ['gfs_mean_117m_ws', 'gfs_mean_117m_ws_cubed', 'ldaps_mean_117m_ws', 'ldaps_mean_117m_ws_cubed']
corr_v = df_full[v_cols + TARGET_COLS].corr().loc[v_cols, TARGET_COLS]

sns.heatmap(corr_v, annot=True, fmt=".4f", cmap="Blues", ax=axes[0, 1])
axes[0, 1].set_title("2) Wind Speed vs Target Correlation")

# 3. v5 신규 도메인 피처 (푄현상, Ramp 차분 등) 상관계수
v5_cols = ['foehn_index_peak', 'heat_stagnation_index', 'ws_diff_prev1', 'ws_rolling_std_3h', 'diff_ws_117m_gfs_ldaps']
corr_v5 = df_full[v5_cols + TARGET_COLS].corr().loc[v5_cols, TARGET_COLS]

sns.heatmap(corr_v5, annot=True, fmt=".4f", cmap="YlOrRd", ax=axes[1, 0])
axes[1, 0].set_title("3) Domain Features (Foehn, Ramp, Model Diff) Correlation")

# 4. GFS 117m vs LDAPS 117m 예보 차이 산점도
axes[1, 1].scatter(df_full['gfs_mean_117m_ws'], df_full['ldaps_mean_117m_ws'], alpha=0.1, color='darkcyan', s=5)
axes[1, 1].plot([0, 30], [0, 30], 'r--', label='1:1 Line')
axes[1, 1].set_title("4) GFS 117m WS vs LDAPS 117m WS Discrepancy")
axes[1, 1].set_xlabel("GFS 117m WS (m/s)")
axes[1, 1].set_ylabel("LDAPS 117m WS (m/s)")
axes[1, 1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# SCADA 데이터 로드
scada_vestas = pd.read_csv(TRAIN_DIR / "scada_vestas_train.csv", encoding='utf-8-sig')
scada_unison = pd.read_csv(TRAIN_DIR / "scada_unison_train.csv", encoding='utf-8-sig')

scada_vestas['kst_dtm'] = pd.to_datetime(scada_vestas['kst_dtm'])
scada_unison['kst_dtm'] = pd.to_datetime(scada_unison['kst_dtm'])

# VESTAS 1호기 풍속 vs 출력 파워 커브 시각화
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].hexbin(
    scada_vestas['vestas_wtg01_ws'], 
    scada_vestas['vestas_wtg01_power_kw10m'], 
    gridsize=60, 
    cmap='viridis', 
    mincnt=1
)
axes[0].set_title("VESTAS WTG01 SCADA Power Curve (Soft Cut-out Check)")
axes[0].set_xlabel("Wind Speed (m/s)")
axes[0].set_ylabel("10min Power (kW10m)")
axes[0].grid(True, alpha=0.3)

# UNISON 1호기 풍속 vs 출력 파워 커브 시각화
axes[1].hexbin(
    scada_unison['unison_wtg01_ws'], 
    scada_unison['unison_wtg01_power_kw10m'], 
    gridsize=60, 
    cmap='magma', 
    mincnt=1
)
axes[1].set_title("UNISON WTG01 SCADA Power Curve (Soft Cut-out Check)")
axes[1].set_xlabel("Wind Speed (m/s)")
axes[1].set_ylabel("10min Power (kW10m)")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()